In [2]:
import os
import random
import json

In [10]:
random.seed(0) 
PAIR_PARTNER = {0: 18, 18: 0, 3: 15, 15: 3, 7: 17, 17: 7}
TRICKY_PAIRS = {frozenset({0, 18}), frozenset({3, 15}), frozenset({7, 17})}

In [11]:
root_dir = './experiment_clusters'
num_examples = 5000
group_size = 5

In [12]:
def is_tricky_pair(a_first, b_first):
    return frozenset({a_first, b_first}) in TRICKY_PAIRS

In [13]:
clusters = {}
for cluster_name in os.listdir(root_dir):

    cluster_path = os.path.join(root_dir, cluster_name)
    if not os.path.isdir(cluster_path):
        continue
    try:
        first, second = cluster_name.split('_')
        first = int(first) if first else None
        second = int(second)
    except ValueError:

        continue
 
    plans_dir = os.path.join(cluster_path, 'floorplan_reoriented')
    if not os.path.isdir(plans_dir):
        continue

    plan_ids = [os.path.splitext(f)[0] for f in os.listdir(plans_dir) if f.endswith('.png')]
    if len(plan_ids) < 4:
        continue
    clusters[(first, second)] = plan_ids


In [14]:
def sample_difficult_group(base_key, all_keys):
    """
    Non-outliers: group_size-1 from *one* subfolder = base_key
    Outlier:      from any subfolder whose first label is the paired counterpart.
    Enforces 'tricky pair' requirement.
    """
    base_first, base_second = base_key
    if base_first not in PAIR_PARTNER:
        return None  # difficult set must use (0,18) (3,15) (7,17) pairs

    base_ids = clusters[base_key]
    if len(base_ids) < group_size - 1:
        return None

    # choose non-outliers from the same subfolder
    non_outliers = random.sample(base_ids, group_size - 1)

    # choose outlier from any subfolder with the *paired* first label
    partner_first = PAIR_PARTNER[base_first]
    candidates = [k for k in all_keys if k[0] == partner_first]
    if not candidates:
        return None
    other_key = random.choice(candidates)
    other_id = random.choice(clusters[other_key])

    # (sanity) the first-label pair must be tricky
    assert is_tricky_pair(base_first, other_key[0])

    return {
        "dataset": "difficult",
        "base_cluster": base_key,
        "other_cluster": other_key,
        "group": non_outliers + [other_id],
        "outlier_id": other_id
    }


def sample_easy_group(base_key, all_keys):
    """
    Non-outliers: group_size-1 from *one* subfolder = base_key
    Outlier:      any subfolder with different first and different second,
                  BUT must NOT form a (0,18), (3,15), or (7,17) pair.
    """
    base_first, base_second = base_key
    base_ids = clusters[base_key]
    if len(base_ids) < group_size - 1:
        return None

    non_outliers = random.sample(base_ids, group_size - 1)

    # "default" structure like your sample_group default,
    # excluding tricky first-label pairs
    candidates = [
        k for k in all_keys
        if k[0] != base_first and k[1] != base_second and not is_tricky_pair(base_first, k[0])
    ]
    if not candidates:
        return None
    other_key = random.choice(candidates)
    other_id = random.choice(clusters[other_key])

    # (sanity) ensure we didn't accidentally include a tricky pair
    assert not is_tricky_pair(base_first, other_key[0])

    return {
        "dataset": "easy",
        "base_cluster": base_key,
        "other_cluster": other_key,
        "group": non_outliers + [other_id],
        "outlier_id": other_id
    }

# --- DATASET BUILDERS -------------------------------------------------------

def build_dataset(clusters, sampler, n_examples, require_base_in_tricky=False):
    """
    Tries to assemble up to n_examples groups using `sampler`.
    If require_base_in_tricky=True, base_key must have a first label in {0,18,3,15,7,17}.
    """
    all_keys = list(clusters.keys())
    results = []
    seen = set()  # avoid exact duplicate groups (ignores order)

    # Pre-filter bases if needed for difficult mode
    if require_base_in_tricky:
        tricky_firsts = {0, 18, 3, 15, 7, 17}
        base_pool = [k for k in all_keys if k[0] in tricky_firsts and len(clusters[k]) >= (group_size - 1)]
    else:
        base_pool = [k for k in all_keys if len(clusters[k]) >= (group_size - 1)]

    if not base_pool:
        return results

    attempts = 0
    max_attempts = 50 * n_examples  # generous cap to avoid infinite loops

    while len(results) < n_examples and attempts < max_attempts:
        attempts += 1
        base_key = random.choice(base_pool)
        sample = sampler(base_key, all_keys)
        if not sample:
            continue

        # De-dup by the image IDs, regardless of order
        key = tuple(sorted(sample["group"]))
        if key in seen:
            continue
        seen.add(key)
        results.append(sample)

    return results

In [15]:
num_easy = num_examples
num_difficult = num_examples

easy_dataset = build_dataset(
    clusters=clusters,
    sampler=sample_easy_group,
    n_examples=num_easy,
    require_base_in_tricky=False
)

difficult_dataset = build_dataset(
    clusters=clusters,
    sampler=sample_difficult_group,
    n_examples=num_difficult,
    require_base_in_tricky=True  # base must come from 0/18/3/15/7/17
)

print(f"Constructed {len(easy_dataset)} easy groups and {len(difficult_dataset)} difficult groups.")

# --- SAVE -------------------------------------------------------------------

out_easy = os.path.join(root_dir, "easy_dataset_mid.json")
out_diff = os.path.join(root_dir, "difficult_dataset_mid.json")

with open(out_easy, "w") as f:
    json.dump(easy_dataset, f, indent=2)

with open(out_diff, "w") as f:
    json.dump(difficult_dataset, f, indent=2)

print(f"Saved:\n  {out_easy}\n  {out_diff}")

Constructed 5000 easy groups and 5000 difficult groups.
Saved:
  ./experiment_clusters/easy_dataset_mid.json
  ./experiment_clusters/difficult_dataset_mid.json


In [9]:
from itertools import combinations
import json
import os

# --- Pairing rules (same as before) -----------------------------------------
PAIR_PARTNER = {0: 18, 18: 0, 3: 15, 15: 3, 7: 17, 17: 7}
TRICKY_PAIRS = {frozenset({0, 18}), frozenset({3, 15}), frozenset({7, 17})}

def is_tricky_pair(a_first, b_first):
    return frozenset({a_first, b_first}) in TRICKY_PAIRS

# --- Streaming JSON array writer (avoids holding everything in memory) ------
class JsonArrayWriter:
    def __init__(self, path):
        self.path = path
        self.f = None
        self.first = True
    def __enter__(self):
        os.makedirs(os.path.dirname(self.path), exist_ok=True)
        self.f = open(self.path, "w", encoding="utf-8")
        self.f.write("[\n")
        return self
    def write_obj(self, obj):
        if not self.first:
            self.f.write(",\n")
        json.dump(obj, self.f)
        self.first = False
    def __exit__(self, exc_type, exc, tb):
        self.f.write("\n]\n")
        self.f.close()

# --- Exhaustive generators ---------------------------------------------------
# Dedup guarantee: we normalize each group's IDs as a sorted tuple and avoid writing
# if we've already seen that exact 5-ID set. (Order within the group doesn’t matter.)
# If you’re sure filenames are globally unique and enumeration won’t collide, you can
# set DEDUP = False for speed & lower memory.
DEDUP = True

def generate_difficult_stream(clusters, writer, group_size=5):
    """
    Difficult: non-outliers = (group_size-1) from one subfolder (base_key),
    outlier from any subfolder whose *first* label is the paired counterpart.
    """
    seen = set()
    count = 0
    tricky_firsts = {0, 18, 3, 15, 7, 17}

    for base_key, base_ids in clusters.items():
        base_first, base_second = base_key
        if base_first not in tricky_firsts:
            continue
        if len(base_ids) < group_size - 1:
            continue

        partner_first = PAIR_PARTNER[base_first]
        partner_keys = [k for k in clusters.keys() if k[0] == partner_first]
        if not partner_keys:
            continue

        # all combinations of non-outliers from this one subfolder
        for non_outliers in combinations(base_ids, group_size - 1):
            for other_key in partner_keys:
                for outlier_id in clusters[other_key]:
                    group = list(non_outliers) + [outlier_id]
                    key = tuple(sorted(group))
                    if DEDUP and key in seen:
                        continue
                    seen.add(key)

                    writer.write_obj({
                        "dataset": "difficult",
                        "base_cluster": base_key,
                        "other_cluster": other_key,
                        "group": group,
                        "outlier_id": outlier_id
                    })
                    count += 1
    return count


def generate_easy_stream(clusters, writer, group_size=5):
    """
    Easy: non-outliers = (group_size-1) from one subfolder (base_key),
    outlier from any subfolder with different first AND different second,
    and whose first label does NOT form a tricky pair with the base first.
    """
    seen = set()
    count = 0

    for base_key, base_ids in clusters.items():
        if len(base_ids) < group_size - 1:
            continue
        base_first, base_second = base_key

        candidates = [
            k for k in clusters.keys()
            if k[0] != base_first
            and k[1] != base_second
            and not is_tricky_pair(base_first, k[0])
        ]
        if not candidates:
            continue

        for non_outliers in combinations(base_ids, group_size - 1):
            for other_key in candidates:
                for outlier_id in clusters[other_key]:
                    group = list(non_outliers) + [outlier_id]
                    key = tuple(sorted(group))
                    if DEDUP and key in seen:
                        continue
                    seen.add(key)

                    writer.write_obj({
                        "dataset": "easy",
                        "base_cluster": base_key,
                        "other_cluster": other_key,
                        "group": group,
                        "outlier_id": outlier_id
                    })
                    count += 1
    return count

# --- Build & save *all possible* unique examples ----------------------------
out_easy = os.path.join(root_dir, "easy_dataset_full.json")
out_diff = os.path.join(root_dir, "difficult_dataset_full.json")

with JsonArrayWriter(out_easy) as easy_writer:
    easy_count = generate_easy_stream(clusters, easy_writer, group_size=group_size)

with JsonArrayWriter(out_diff) as diff_writer:
    diff_count = generate_difficult_stream(clusters, diff_writer, group_size=group_size)

print(f"Constructed {easy_count} easy and {diff_count} difficult unique groups.")
print(f"Saved:\n  {out_easy}\n  {out_diff}")

Constructed 4901220 easy and 808801 difficult unique groups.
Saved:
  ./experiment_clusters/easy_dataset_full.json
  ./experiment_clusters/difficult_dataset_full.json
